# ProbJAX PPL Basics

This notebook uses the probabilistic-first API names from `probjax.core`: `observe`, `do`, and `log_joint_fn` (with backward-compatible aliases still available).

In [ ]:
import jax
import jax.numpy as jnp

from probjax.core import do, joint_sample, log_joint_fn, observe, scope, substitute, trace
from probjax.core.custom_primitives.random_variable import rv_p
from probjax.stats import norm

In [ ]:
def model(key):
    k1, k2 = jax.random.split(key)
    z = rv_p.bind(k1, 0.0, 1.0, dist=norm, name="z")
    y = rv_p.bind(k2, z, 0.5, dist=norm, name="y")
    return y

## 1) Joint sampling

In [ ]:
joint_sample(model)(jax.random.PRNGKey(0))

## 2) Observe data and evaluate log-joint

In [ ]:
obs_y = jnp.array(0.25)
observed_model = observe(model, {"y": obs_y})
latent_samples = joint_sample(observed_model)(jax.random.PRNGKey(1))
log_joint_fn(observed_model)(z=latent_samples["z"])

## 3) Substitute in `condition` mode vs `do` intervention

In [ ]:
fixed_z = jnp.array(-0.4)

conditioned_substitute = substitute(model, {"z": fixed_z}, mode="condition")
conditioned_samples = joint_sample(conditioned_substitute)(jax.random.PRNGKey(2))
conditioned_log_joint = log_joint_fn(conditioned_substitute)(y=conditioned_samples["y"])

do_model = do(model, {"z": fixed_z})
do_samples = joint_sample(do_model)(jax.random.PRNGKey(3))
do_log_joint = log_joint_fn(do_model)(y=do_samples["y"])

{
    "conditioned_samples": conditioned_samples,
    "conditioned_log_joint": conditioned_log_joint,
    "do_samples": do_samples,
    "do_log_joint": do_log_joint,
}

`mode="condition"` keeps the substituted site's density term. `do(...)` removes the intervened site's own density term.

## 4) Site-aware tracing

In [ ]:
trace(model, sites=True)(jax.random.PRNGKey(4))

In [ ]:
trace(conditioned_substitute, sites=True)(jax.random.PRNGKey(5))

In [ ]:
trace(do_model, sites=True)(jax.random.PRNGKey(6))

## 5) Strict vs partial log-joint evaluation

In [ ]:
strict_log_joint = log_joint_fn(model)
try:
    strict_log_joint(z=jnp.array(0.1))
except KeyError as err:
    print("strict mode error:", err)

In [ ]:
partial_log_joint = log_joint_fn(model, allow_partial=True)
partial_log_joint(z=jnp.array(0.1))

## 6) Scoped auto-generated site names

In [ ]:
def scoped_model(key):
    with scope("outer"):
        z = rv_p.bind(key, 0.0, 1.0, dist=norm)
    return z

joint_sample(scoped_model)(jax.random.PRNGKey(7))